In [1]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# import seaborn as sns
# import matplotlib.pyplot as plt
from scipy.stats import entropy
import pandas as pd
import os

In [3]:
# Load CSV files
data_dir = '../../mcphases/'

# active_minutes = pd.read_csv(os.path.join(data_dir, 'active_minutes.csv'))
# calories = pd.read_csv(os.path.join(data_dir, 'calories.csv'))
# demographic_vo2_max = pd.read_csv(os.path.join(data_dir, 'demographic_vo2_max.csv'))
# exercise = pd.read_csv(os.path.join(data_dir, 'exercise.csv'))
# time_in_heart_rate_zones = pd.read_csv(os.path.join(data_dir, 'time_in_heart_rate_zones.csv'))
# height_and_weight = pd.read_csv(os.path.join(data_dir, 'height_and_weight.csv'))
# subject_info = pd.read_csv(os.path.join(data_dir, 'subject-info.csv'))
hormones_and_selfreport = pd.read_csv(os.path.join(data_dir, 'hormones_and_selfreport.csv'))

print("All CSV files loaded successfully!")

All CSV files loaded successfully!


In [4]:
hormones_and_selfreport.head()

,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,flow_color,...,headaches,cramps,sorebreasts,fatigue,sleepissue,moodswing,stress,foodcravings,indigestion,bloating
0,1,2022,True,1,Follicular,2.9,94.2,NaN,Not at all,Not at all,...,High,Very Low/Little,Very Low/Little,High,Low,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
1,1,2022,False,2,Follicular,1.2,226.3,NaN,Not at all,Not at all,...,Very High,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
2,1,2022,False,3,Follicular,3.5,276.8,NaN,Not at all,Not at all,...,High,Very Low/Little,Very Low/Little,Very High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
3,1,2022,False,4,Fertility,1.8,322.1,NaN,Not at all,Not at all,...,Very Low/Little,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
4,1,2022,False,5,Fertility,4.6,244.9,NaN,Not at all,Not at all,...,Very Low/Little,Very Low/Little,Very Low/Little,High,High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little


### 0. convert Likert to numerical

In [5]:
# Standard symptom scale
symptom_likert_map = {
    'Not at all': 0,
    'Very Low/Little': 1,
    'Very Low': 1,
    'Low': 2,
    'Moderate': 3,
    'High': 4,
    'Very High': 5,
}

symptoms = [
    'appetite', 'headaches', 'cramps', 'sorebreasts', 'fatigue',
    'sleepissue', 'moodswing', 'stress', 'foodcravings',
    'indigestion', 'bloating', 'exerciselevel'
]

for col in symptoms:
    hormones_and_selfreport[col + '_num'] = hormones_and_selfreport[col].map(symptom_likert_map)


In [6]:
# Should return empty if all labels mapped
for col in symptoms + ['exerciselevel']:
    unmapped = hormones_and_selfreport.loc[
        hormones_and_selfreport[col].notna() &
        hormones_and_selfreport[col + '_num'].isna(),
        col
    ].unique()
    print(f"Unmapped values for {col}: {unmapped}")

Unmapped values for appetite: []
Unmapped values for headaches: ['2' '5' '3' '4']
Unmapped values for cramps: []
Unmapped values for sorebreasts: []
Unmapped values for fatigue: []
Unmapped values for sleepissue: []
Unmapped values for moodswing: []
Unmapped values for stress: ['2' '3' '1']
Unmapped values for foodcravings: []
Unmapped values for indigestion: []
Unmapped values for bloating: []
Unmapped values for exerciselevel: []
Unmapped values for exerciselevel: []


### 1. Check for extreme distribution of features

In [7]:
# checking for floor/ceiling effects in self-reported symptoms
new_symptoms = [col + '_num' for col in symptoms]

for symptom in new_symptoms:
    print(hormones_and_selfreport[symptom].value_counts(normalize=True).sort_index())

appetite_num
0.0    0.001201
1.0    0.045345
2.0    0.227628
3.0    0.524925
4.0    0.174174
5.0    0.026727
Name: proportion, dtype: float64
headaches_num
0.0    0.327711
1.0    0.266867
2.0    0.151807
3.0    0.151205
4.0    0.076506
5.0    0.025904
Name: proportion, dtype: float64
cramps_num
0.0    0.473700
1.0    0.281635
2.0    0.088668
3.0    0.096183
4.0    0.042080
5.0    0.017734
Name: proportion, dtype: float64
sorebreasts_num
0.0    0.516381
1.0    0.265104
2.0    0.105200
3.0    0.087466
4.0    0.017433
5.0    0.008416
Name: proportion, dtype: float64
fatigue_num
0.0    0.133293
1.0    0.141999
2.0    0.165716
3.0    0.280997
4.0    0.205644
5.0    0.072351
Name: proportion, dtype: float64
sleepissue_num
0.0    0.198859
1.0    0.194953
2.0    0.230400
3.0    0.206669
4.0    0.108741
5.0    0.060378
Name: proportion, dtype: float64
moodswing_num
0.0    0.310843
1.0    0.234940
2.0    0.163855
3.0    0.184337
4.0    0.080723
5.0    0.025301
Name: proportion, dtype: float64
st

Note: There are some numerical values (before the conversion) in stress and headaches might indicate the change of format in data collection. However, they are a minimal proportion of the dataset. So, probably can leave it as is since I'm going to convert the text ordinal to numerical later.

Looking at possible floor/ceiling effects:
**fatigue, stress, sleepissue, foodcravings** are best distributed.

### 2. Examine the variation in the features

In [8]:
# Analyze the distribution of numeric symptoms and calculate entropy
results = []

for col in new_symptoms:

    numeric = hormones_and_selfreport[col]

    probs = (
        numeric
        .value_counts(normalize=True)
        .sort_index()
    )

    results.append({
        'symptom': col,
        'missing_pct': numeric.isna().mean() * 100,
        'std': numeric.std(),
        'entropy': entropy(probs)
    })

results = pd.DataFrame(results)

results.sort_values(
    ['std','entropy'],
    ascending=False
)

,symptom,missing_pct,std,entropy
8,foodcravings_num,41.208694,1.526374,1.688091
10,bloating_num,41.191023,1.478250,1.622679
4,fatigue_num,41.138010,1.476416,1.715616
5,sleepissue_num,41.173352,1.475101,1.714754
7,stress_num,41.332391,1.465387,1.700836
6,moodswing_num,41.332391,1.422916,1.607780
9,indigestion_num,41.244036,1.416349,1.574793
1,headaches_num,41.332391,1.405944,1.581243
2,cramps_num,41.208694,1.268866,1.355679
3,sorebreasts_num,41.208694,1.113906,1.254055


High-variation features: foodcravings, bloating, fatigue, sleepissue, stress, moodswing, indigestion, headaches, cramps, sorebreasts.

In [9]:
# Examine how symptoms vary across menstrual cycle phases
symptoms = [
    'headaches_num',
    'cramps_num',
    'sorebreasts_num',
    'fatigue_num',
    'sleepissue_num',
    'moodswing_num',
    'stress_num',
    'foodcravings_num',
    'bloating_num',
    'indigestion_num'
]

phase_means = (
    hormones_and_selfreport
    .groupby('phase')[symptoms]
    .mean()
)

print(phase_means)

            headaches_num  cramps_num  sorebreasts_num  fatigue_num  \
phase                                                                 
Fertility        1.363388    0.762295         0.665301     2.395095   
Follicular       1.543504    0.641667         0.552381     2.489893   
Luteal           1.318386    0.933751         0.949866     2.469589   
Menstrual        1.710900    1.886970         1.279435     2.695447   

            sleepissue_num  moodswing_num  stress_num  foodcravings_num  \
phase                                                                     
Fertility         2.047814       1.478142    2.549180          1.695355   
Follicular        1.951249       1.474374    2.601907          1.804762   
Luteal            1.980322       1.518386    2.502242          1.820949   
Menstrual         2.113030       1.870458    2.677725          2.023548   

            bloating_num  indigestion_num  
phase                                      
Fertility       1.517760         1

**Overall**, fatigue, stress, sleepissue, and foodcravings are best targets if we would like to train regression models.